# Phase 1: Problem Formulation & Analytical Framework
**Financial Market Analytics & Investment Decision Support System for Indian Equities**
*CRISP-DM Phase: Business & Problem Understanding*

---

### Educational & Analytical Context
This project implements an end-to-end investment decision-support and market analytics system covering 10 years of Indian equity market data (NSE). 

> **Important Regulatory & Educational Notice:**  
> This application is strictly an educational decision-support tool designed to demonstrate quantitative finance, portfolio analytics, and explainable decision-support systems. It is **not** SEBI-registered investment advice and **not** a trading terminal. All UI and analytical outputs intentionally avoid directive "buy/sell" language.


## 1. Problem Statement & Sub-Problems (PRD §2)

### Core Problem
Retail equity investors in India have access to vast amounts of raw data (price charts, daily quotes, financial media opinions), but lack a structured, statistically sound framework to:
1. Compare companies and sectors on a standardized, risk-adjusted basis.
2. Quantify multidimensional investment risk (volatility regimes, drawdown depth, tail risk, and fat-tailed return distributions).
3. Translate objective historical statistics into personalized, explainable guidance tailored to an individual investor's risk temperament and existing portfolio.

### Sub-Problems
1. **Like-for-Like Comparison:** Heterogeneous price levels prevent direct comparison without rebasing or standardizing cumulative return series.
2. **Comprehensive Risk Evaluation:** Standard deviations alone fail to capture tail risk, crash frequency, and prolonged recovery periods in Indian equities.
3. **Decision-Support Gap:** Analytical platforms often stop at raw data visualization without answering *"what does this statistic mean for a retail investor?"*
4. **Personalization & Explainability:** Commercial robo-advisors often operate as black-box algorithms. Investors require transparent, auditable rationales where every recommendation links directly to an inspectable statistic.


## 2. Project Goals & Non-Goals

### Goals
- **G1 (Reproducible Analytics):** Answer core analytical questions (Q1–Q13) with mathematically verified, unit-tested statistical methods.
- **G2 (Risk Profiling):** Evaluate risk capacity and tolerance via a psychometrically grounded 5-tier risk questionnaire (PRD §11).
- **G3 (Portfolio Health Analysis):** Diagnose user portfolios for sector concentration, cap-class imbalance, weighted volatility, and average correlation.
- **G4 (Explainable Recommendation):** Score and rank candidate equities using transparent multi-factor scoring (Sharpe ratio, drawdown resilience, diversification bonus, sector balance).
- **G5 (Production Deployment):** Deliver a resilient, read-only Streamlit web application powered by DuckDB with automated CI pipeline refreshes.

### Non-Goals (Scope Boundaries for v1)
- **No Intraday / Tick-Level Trading:** Long-term investment decisions rely on settled daily closes; tick data introduces noise without analytical value.
- **No Order Execution / Broker APIs:** No automated order routing or trading account connectivity.
- **No Black-Box Price Forecasting:** Recommendations do not rely on speculative neural network price predictions.
- **No Storage of User PII:** User portfolios and questionnaire responses reside purely in session memory (`st.session_state`) and are never written to disk or third-party servers.


## 3. Analytical Questions Framework (PRD §3)

The project answers 13 formal analytical questions. Each question defines a concrete metric, mathematical method, and practical decision-support outcome across the project lifecycle:


In [1]:
import pandas as pd

questions_data = [
    {
        "#": "Q1",
        "Analytical Question": "How big is the dataset and what is the typical market return?",
        "Metric / Method": "Row/ticker counts, trading calendar coverage, mean daily simple return, 10-year CAGR",
        "Decision / Hypothesis": "Establish baseline market KPIs and historical growth trajectory.",
        "CRISP-DM Phase": "EDA (Phase 4)"
    },
    {
        "#": "Q2",
        "Analytical Question": "Which sectors grew most over the 10-year horizon?",
        "Metric / Method": "Equal-weighted rebased price (base 100 on 2016-01-01), annualized CAGR by sector",
        "Decision / Hypothesis": "Identify structural long-term sectoral winners vs laggards.",
        "CRISP-DM Phase": "EDA (Phase 4)"
    },
    {
        "#": "Q3",
        "Analytical Question": "Which sectors provide superior risk-adjusted returns?",
        "Metric / Method": "Annualized Sharpe Ratio, Sortino Ratio (downside deviation, Rf=6.5%)",
        "Decision / Hypothesis": "Determine return efficiency per unit of total and downside risk.",
        "CRISP-DM Phase": "Stats (Phase 5)"
    },
    {
        "#": "Q4",
        "Analytical Question": "How has an individual stock behaved across cycles (growth, crash, recovery)?",
        "Metric / Method": "Rebased price trajectory, underwater drawdown curve, maximum drawdown, peak-to-recovery time",
        "Decision / Hypothesis": "Stock deep-dive profiling and recovery persistence analysis.",
        "CRISP-DM Phase": "EDA / Stock Deep Dive"
    },
    {
        "#": "Q5",
        "Analytical Question": "Are price movements confirmed by trading volume?",
        "Metric / Method": "Volume vs absolute daily return correlation, 20-day rolling volume z-scores",
        "Decision / Hypothesis": "Distinguish high-conviction institutional moves from low-volume noise.",
        "CRISP-DM Phase": "EDA (Phase 4)"
    },
    {
        "#": "Q6",
        "Analytical Question": "How risky is a stock on a day-to-day basis?",
        "Metric / Method": "Return distributions, skewness, excess kurtosis, 90-day rolling annualized volatility, Historical VaR 95%",
        "Decision / Hypothesis": "Quantify tail risk and volatility regime shifts.",
        "CRISP-DM Phase": "EDA / Stats"
    },
    {
        "#": "Q7",
        "Analytical Question": "Which stocks move together and where are diversification benefits?",
        "Metric / Method": "Pairwise correlation matrix (80 stocks), hierarchical clustering, calm vs crash correlation shifts",
        "Decision / Hypothesis": "Avoid redundant holdings and prevent systemic diversification collapse.",
        "CRISP-DM Phase": "EDA (Phase 4)"
    },
    {
        "#": "Q8",
        "Analytical Question": "Is the return generated worth the systematic risk taken?",
        "Metric / Method": "Capital Asset Pricing Model (CAPM) Beta against Nifty 50 (^NSEI), Jensen's Alpha, Treynor Ratio",
        "Decision / Hypothesis": "Distinguish genuine market outperformance from levered beta exposure.",
        "CRISP-DM Phase": "Stats (Phase 5)"
    },
    {
        "#": "Q9",
        "Analytical Question": "Are sectors and cap classes statistically significantly different in risk?",
        "Metric / Method": "Kruskal-Wallis non-parametric test, Levene test for homogeneity of variance",
        "Decision / Hypothesis": "H1: Large-cap equities exhibit statistically lower volatility than small caps.",
        "CRISP-DM Phase": "Stats (Phase 5)"
    },
    {
        "#": "Q10",
        "Analytical Question": "How do equities behave during severe market crises?",
        "Metric / Method": "Crisis event studies: COVID-19 Crash (Feb-Mar 2020), 2022 Inflation Shock; Downside Beta",
        "Decision / Hypothesis": "Classify resilient defensive sectors vs highly vulnerable cyclicals.",
        "CRISP-DM Phase": "EDA (Phase 4)"
    },
    {
        "#": "Q11",
        "Analytical Question": "What is this specific investor's risk profile?",
        "Metric / Method": "Two-dimensional psychometric score (Financial Capacity + Psychological Tolerance) mapped to Profiles 1-5",
        "Decision / Hypothesis": "Establish target equity/debt/cash allocation and cap-class bounds.",
        "CRISP-DM Phase": "Profile (Phase 6)"
    },
    {
        "#": "Q12",
        "Analytical Question": "How diversified is the investor's current portfolio?",
        "Metric / Method": "Herfindahl-Hirschman Index (HHI), max single-stock/sector weights, average pairwise correlation",
        "Decision / Hypothesis": "Identify concentration vulnerabilities exceeding PRD limits (stock 25%, sector 40%).",
        "CRISP-DM Phase": "Portfolio (Phase 7)"
    },
    {
        "#": "Q13",
        "Analytical Question": "Which candidate stocks improve portfolio quality and fit the risk profile?",
        "Metric / Method": "Multi-factor explainable rule-based scoring (Sharpe 35%, Drawdown 20%, Diversification 25%, Sector Gap 10%, Momentum 10%)",
        "Decision / Hypothesis": "Produce transparent, ranked recommendations with detailed rationale cards.",
        "CRISP-DM Phase": "Recommend (Phase 8)"
    }
]

df_q = pd.DataFrame(questions_data)
# Display formatted table
pd.set_option('display.max_colwidth', None)
df_q[['#', 'Analytical Question', 'Metric / Method', 'Decision / Hypothesis']]


,#,Analytical Question,Metric / Method,Decision / Hypothesis
0,Q1,How big is the dataset and what is the typical market return?,"Row/ticker counts, trading calendar coverage, mean daily simple return, 10-year CAGR",Establish baseline market KPIs and historical growth trajectory.
1,Q2,Which sectors grew most over the 10-year horizon?,"Equal-weighted rebased price (base 100 on 2016-01-01), annualized CAGR by sector",Identify structural long-term sectoral winners vs laggards.
2,Q3,Which sectors provide superior risk-adjusted returns?,"Annualized Sharpe Ratio, Sortino Ratio (downside deviation, Rf=6.5%)",Determine return efficiency per unit of total and downside risk.
3,Q4,"How has an individual stock behaved across cycles (growth, crash, recovery)?","Rebased price trajectory, underwater drawdown curve, maximum drawdown, peak-to-recovery time",Stock deep-dive profiling and recovery persistence analysis.
4,Q5,Are price movements confirmed by trading volume?,"Volume vs absolute daily return correlation, 20-day rolling volume z-scores",Distinguish high-conviction institutional moves from low-volume noise.
5,Q6,How risky is a stock on a day-to-day basis?,"Return distributions, skewness, excess kurtosis, 90-day rolling annualized volatility, Historical VaR 95%",Quantify tail risk and volatility regime shifts.
6,Q7,Which stocks move together and where are diversification benefits?,"Pairwise correlation matrix (80 stocks), hierarchical clustering, calm vs crash correlation shifts",Avoid redundant holdings and prevent systemic diversification collapse.
7,Q8,Is the return generated worth the systematic risk taken?,"Capital Asset Pricing Model (CAPM) Beta against Nifty 50 (^NSEI), Jensen's Alpha, Treynor Ratio",Distinguish genuine market outperformance from levered beta exposure.
8,Q9,Are sectors and cap classes statistically significantly different in risk?,"Kruskal-Wallis non-parametric test, Levene test for homogeneity of variance",H1: Large-cap equities exhibit statistically lower volatility than small caps.
9,Q10,How do equities behave during severe market crises?,"Crisis event studies: COVID-19 Crash (Feb-Mar 2020), 2022 Inflation Shock; Downside Beta",Classify resilient defensive sectors vs highly vulnerable cyclicals.


## 4. System Configuration & Parameter Governance (Hard Rule 5)

Per **AGENTS.md Hard Rule 5**:
> *All thresholds, weights, dates, the risk-free rate, and the stock universe live in `config.yaml` (and `universe.csv`). No magic numbers in code.*

Below, we load the validated application configuration via `src.config.load_config()` to inspect the analytical constants governing the platform:


In [2]:
import sys
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import load_config

config = load_config(project_root / "config.yaml")

print("=== SYSTEM CONFIGURATION SUMMARY ===")
print(f"Date Range:              {config.date_range.start} to {config.date_range.end}")
print(f"Risk-Free Rate (Rf):     {config.risk_free_rate:.1%}")
print(f"Trading Days / Year:     {config.trading_days}")
print(f"Random Seed:             {config.random_seed}")
print(f"Extreme Z Threshold:     {config.extreme_z_threshold}")
print(f"Spike Review Threshold:  {config.spike_review_threshold:.0%}")
print(f"Min History Years:       {config.min_history_years}")
print(f"Min Liquidity Traded:    INR {config.min_liquidity_traded_value:,.0f}")
print("\n--- Concentration Limits ---")
print(f"Max Single Stock Weight: {config.concentration_limits.stock:.1%}")
print(f"Max Single Sector Weight:{config.concentration_limits.sector:.1%}")
print("\n--- Recommender Weights ---")
print(f"Sharpe Ratio Weight:     {config.recommender_weights.sharpe:.1%}")
print(f"Drawdown Score Weight:   {config.recommender_weights.drawdown:.1%}")
print(f"Diversification Weight:  {config.recommender_weights.diversification:.1%}")
print(f"Sector Gap Weight:       {config.recommender_weights.sector_gap:.1%}")
print(f"Dividend/Momentum Weight:{config.recommender_weights.dividend_or_momentum:.1%}")
print(f"Total Weights Sum:       {sum([config.recommender_weights.sharpe, config.recommender_weights.drawdown, config.recommender_weights.diversification, config.recommender_weights.sector_gap, config.recommender_weights.dividend_or_momentum]):.2f}")


=== SYSTEM CONFIGURATION SUMMARY ===
Date Range:              2016-01-01 to latest
Risk-Free Rate (Rf):     6.5%
Trading Days / Year:     252
Random Seed:             42
Extreme Z Threshold:     4.0
Spike Review Threshold:  20%
Min History Years:       3
Min Liquidity Traded:    INR 10,000,000

--- Concentration Limits ---
Max Single Stock Weight: 25.0%
Max Single Sector Weight:40.0%

--- Recommender Weights ---
Sharpe Ratio Weight:     35.0%
Drawdown Score Weight:   20.0%
Diversification Weight:  25.0%
Sector Gap Weight:       10.0%
Dividend/Momentum Weight:10.0%
Total Weights Sum:       1.00


In [3]:
# Display Target Allocations for Investor Profiles 1 to 5 (PRD Section 11)
target_alloc_rows = []
for level in sorted(config.target_allocations.keys()):
    alloc = config.target_allocations[level]
    target_alloc_rows.append({
        "Profile Level": level,
        "Profile Label": alloc.label,
        "Equity Target": f"{alloc.equity:.0%}",
        "Debt Target": f"{alloc.debt:.0%}",
        "Gold Target": f"{alloc.gold:.0%}",
        "Large Cap Split": f"{alloc.equity_split.large:.0%}",
        "Mid Cap Split": f"{alloc.equity_split.mid:.0%}",
        "Small Cap Split": f"{alloc.equity_split.small:.0%}"
    })

df_alloc = pd.DataFrame(target_alloc_rows)
df_alloc


,Profile Level,Profile Label,Equity Target,Debt Target,Gold Target,Large Cap Split,Mid Cap Split,Small Cap Split
0,1,Conservative,20%,70%,10%,100%,0%,0%
1,2,Moderately Conservative,35%,55%,10%,90%,10%,0%
2,3,Moderate,50%,40%,10%,70%,25%,5%
3,4,Moderately Aggressive,70%,25%,5%,55%,30%,15%
4,5,Aggressive,85%,10%,5%,40%,35%,25%


In [4]:
# Programmatic Verification of Constraints
assert config.date_range.start == "2016-01-01", "Start date must be 2016-01-01"
assert config.risk_free_rate == 0.065, "Risk free rate must be 6.5%"
assert config.trading_days == 252, "Annualization factor must be 252"
assert config.extreme_z_threshold == 4.0, "Extreme z threshold must be 4"
assert abs(config.recommender_weights.sharpe + 
           config.recommender_weights.drawdown + 
           config.recommender_weights.diversification + 
           config.recommender_weights.sector_gap + 
           config.recommender_weights.dividend_or_momentum - 1.0) < 1e-6, "Weights must sum to 1.0"

for level, alloc in config.target_allocations.items():
    assert abs(alloc.equity + alloc.debt + alloc.gold - 1.0) < 1e-6, f"Profile {level} asset allocation must sum to 1.0"
    assert abs(alloc.equity_split.large + alloc.equity_split.mid + alloc.equity_split.small - 1.0) < 1e-6, f"Profile {level} equity split must sum to 1.0"

print("All configuration constraints and PRD specifications programmatically verified.")


All configuration constraints and PRD specifications programmatically verified.


## 5. Summary & Transition to Data Collection (Phase 2)

### Key Takeaways
1. **Mathematical Clarity:** All 13 analytical questions have rigorous statistical formulations avoiding vague heuristics.
2. **Transparent Parameter Governance:** No magic numbers exist in the codebase. All thresholds (outlier boundaries, concentration limits, recommender weights) are declared in `config.yaml`.
3. **CRISP-DM Alignment:** Business understanding is fully translated into operational requirements, transitioning next to **Phase 2: Data Collection & Universe Curation** (`notebooks/02_data_collection.ipynb`).
